In [ ]:
from pathlib import Path
from typing import Literal
import colorsys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sample_type: Literal["cesium", "proton"] = "cesium"
if sample_type == "cesium":
    dir_base = Path("/Users/sylvi/topo_data/dna_damage_cache")
elif sample_type == "proton":
    dir_base = Path("/Users/sylvi/topo_data/dna_damage_proton_cache")
else:
    raise ValueError(f"Unknown sample type: {sample_type}")

# plotting
if sample_type == "cesium":
    folder_to_labels = {
        "Controls/nicked": "Control nk",
        "Controls/supercoiled": "Control sc",
        "MilliQ/5_percent_damage": "MilliQ 5%",
        "MilliQ/20_percent_damage": "MilliQ 20%",
        "MilliQ/50_percent_damage": "MilliQ 50%",
        "TE/5_percent_damage": "TE 5%",
        "TE/20_percent_damage": "TE 20%",
        "TE/50_percent_damage": "TE 50%",
    }
    folder_to_colours_hsv = {
        "Control nk": (0 / 360, 0 / 100, 35 / 100),
        "Control sc": (0 / 360, 0 / 100, 70 / 100),
        "MilliQ 5%": (0 / 360, 40 / 100, 40 / 100),
        "MilliQ 20%": (0 / 360, 60 / 100, 60 / 100),
        "MilliQ 50%": (0 / 360, 80 / 100, 80 / 100),
        "TE 5%": (183 / 360, 40 / 100, 40 / 100),
        "TE 20%": (183 / 360, 60 / 100, 60 / 100),
        "TE 50%": (183 / 360, 80 / 100, 80 / 100),
    }
    folder_to_colours_rgb = {folder: colorsys.hsv_to_rgb(*hsv) for folder, hsv in folder_to_colours_hsv.items()}
    plotting_folder_order = [
        "Control nk",
        "Control sc",
        "MilliQ 5%",
        "MilliQ 20%",
        "MilliQ 50%",
        "TE 5%",
        "TE 20%",
        "TE 50%",
    ]
elif sample_type == "proton":
    # - highlet/MQ/100gy: 93
    # - highlet/MQ/50gy: 65
    # - highlet/TE/100gy: 100
    # - highlet/TE/50gy: 60
    # - lowlet/MQ/100gy: 70
    # - lowlet/MQ/50gy: 68
    # - lowlet/TE/100gy: 84
    # - lowlet/TE/50gy: 70
    folder_to_labels = {
        "highlet/MQ/100gy": "High LET MQ 100 Gy",
        "highlet/MQ/50gy": "High LET MQ 50 Gy",
        "highlet/TE/100gy": "High LET TE 100 Gy",
        "highlet/TE/50gy": "High LET TE 50 Gy",
        "lowlet/MQ/100gy": "Low LET MQ 100 Gy",
        "lowlet/MQ/50gy": "Low LET MQ 50 Gy",
        "lowlet/TE/100gy": "Low LET TE 100 Gy",
        "lowlet/TE/50gy": "Low LET TE 50 Gy",
    }
    folder_to_colours_hsv = {
        "High LET MQ 100 Gy": (0 / 360, 40 / 100, 40 / 100),
        "High LET MQ 50 Gy": (0 / 360, 60 / 100, 60 / 100),
        "High LET TE 100 Gy": (183 / 360, 40 / 100, 40 / 100),
        "High LET TE 50 Gy": (183 / 360, 60 / 100, 60 / 100),
        "Low LET MQ 100 Gy": (0 / 360, 80 / 100, 80 / 100),
        "Low LET MQ 50 Gy": (0 / 360, 90 / 100, 90 / 100),
        "Low LET TE 100 Gy": (183 / 360, 80 / 100, 80 / 100),
        "Low LET TE 50 Gy": (183 / 360, 90 / 100, 90 / 100),
    }
    folder_to_colours_rgb = {folder: colorsys.hsv_to_rgb(*hsv) for folder, hsv in folder_to_colours_hsv.items()}
    plotting_folder_order = [
        "High LET MQ 50 Gy",
        "High LET MQ 100 Gy",
        "High LET TE 50 Gy",
        "High LET TE 100 Gy",
        "Low LET MQ 50 Gy",
        "Low LET MQ 100 Gy",
        "Low LET TE 50 Gy",
        "Low LET TE 100 Gy",
    ]

In [ ]:
# load the data
# load the existing analysis results
grain_defect_data_df = pd.read_csv(dir_base / "analysis_results" / "defect_grain_statistics.csv")
print(
    f"Loaded {len(grain_defect_data_df)} rows of defect grain statistics data from {dir_base / 'analysis_results' / 'defect_grain_statistics.csv'}"
)
grain_defect_data_df.head()

# load the tagged data
tagged_data_df = pd.read_csv(dir_base / "analysis_results" / "tagged-output" / "image_tags.csv")
print(
    f"Loaded {len(tagged_data_df)} rows of tagged image statistics data from {dir_base / 'analysis_results' / 'tagged-output' / 'image_tags.csv'}"
)
tagged_data_df.head()
# Get the grain number for each tagged row from the last number in the filename
tagged_data_df["grain_id"] = tagged_data_df["filename"].apply(lambda x: int(x.split("_")[-1].split(".")[0]))

# Merge the two dataframes on the grain_id column
df = pd.merge(tagged_data_df, grain_defect_data_df, on="grain_id", how="inner")

# tag with having any beak if any of the beak columns are True
df["has_any_beak"] = df[["beak_pinch", "beak_full_merge", "beak_partial_merge", "hinge"]].any(axis=1)

print(df.columns)

In [ ]:
# Plot stuff
# For each column, plot a bar for each of the folder_to_labels, with the height of the bar being the number of true values in that column
cols_to_plot = [
    "has_any_beak",
    "beak_pinch",
    "beak_full_merge",
    "beak_partial_merge",
    "hinge",
    "bad_tracing",
    "double_strand_break",
]
for column in cols_to_plot:
    print(f"plotting {column}")

    counts_per_folder = df.groupby("folder_x")[column].sum()

    # normalise by number of rows in each folder
    counts_per_folder = counts_per_folder / df.groupby("folder_x").size()

    # reorder the counts_per_folder to match the plotting_folder_order
    counts_per_folder = counts_per_folder.reindex(plotting_folder_order)

    print(f"Counts for {column}:")
    print(counts_per_folder)

    # plot bar chart for each folder in the correct order
    plt.figure(figsize=(10, 6))
    sns.barplot(
        x=counts_per_folder.index,
        y=counts_per_folder.values,
    )
    plt.xticks(rotation=45, ha="right")
    plt.title(f"Counts of {column} per folder")
    plt.ylabel(f"Fraction of grains with {column}")
    plt.xlabel("Folder")
    plt.tight_layout()

In [ ]:
print(df["double_strand_break"].value_counts())